# Analyse Elmer/Ice runs against observations

Clean, reorganized analysis notebook (rebuilt from the old `CompareModVsObs.ipynb`
scratchpad — the original is kept untouched).

**§0 — Shared setup** loads the mesh, Mouginot catchments, observations and the
per-member diagnostic machinery (mass budget, GL discharge, calving).

**§1 — Single / averaged member vs obs.** `MEMBER_KIND` picks the format:
`"ocx"` (default) analyses the **Mahti OCX C011 production run** (1991-2025);
`"smallenstrans"` analyses one `MEMBER_FILE` from the 96-member ensemble (or its
mean, `USE_AVERAGE = True`). The two formats differ (split files, field names,
mesh) — §0 normalizes both to the same downstream code.
  - *1a Global*: ice discharge & mass change **by catchment** vs observations.
  - *1b 2D*: **velocity** and **dH/dt** change maps vs observations.

**§2 — Ensemble by characteristics.** Compare members grouped by their design
parameters (`inv, ocean, gl_melt, fric_law, n_hyp, c_ext`): parameter-effect
spread plots, **Sobol** sensitivity indices, and ensemble-vs-obs envelopes.
This section needs a full multi-member ensemble (the 96-member SmallEnsTrans set
now; a future OCX small ensemble in Phase 3).

## §0 — Shared setup

Run every cell in this section once. It is common to Parts 1 and 2.

### 0.1  Configuration — all paths in one place

In [ ]:
# ── EDIT ME ─────────────────────────────────────────────────────────────────
from pathlib import Path

# Where the ElmerUgrid tools live (registers the `.ugrid.to_netcdf_forpv` accessor).
POSTPRO_DIR = "/home/jagereli/Postdoc/Data/postpro/elmerugrid"   # local
# On Mahti this was: "/scratch/project_2000339/jagereli/postpro/"

# --- Observations ------------------------------------------------------------
OBS_ISMIP_FILE = "../DATA/AntarcticaObsISMIP7-v1.2.nc"   # velocity / dhdt / bedmachine
# Per-catchment discharge obs (Rignot/IMBIE, BMHF14). Not on this disk yet —
# global-discharge-vs-obs plots are skipped gracefully if this file is missing.
OBS_DISCHARGE_FILE = "../DATA/AIS_discharge_BMHF14.nc"

# --- Ensemble (SmallEnsTrans; swap for the OCX ensemble in Phase 3) ----------
MEMBER_DIR   = Path("/media/jagereli/Expansion1/TLC_ISMIP7_ANT/SmallEnsTrans/RUN3/")
MEMBER_GLOB  = "*/*t_aa_trans*.nc"

# --- Part 1 single member ----------------------------------------------------
# "ocx"  -> the Mahti OCX C011 production run (OCX_STATES_FILE/OCX_FORCING_FILE
#           below). "smallenstrans" -> one member from MEMBER_DIR (MEMBER_FILE
#           picks which; None -> first found).
MEMBER_KIND  = "ocx"        # "ocx" or "smallenstrans"
MEMBER_FILE  = None         # smallenstrans only; None -> first ensemble member
MEMBER_GLMELT = "fmp"       # smallenstrans only: BMB treatment ('fmp' or 'nmp')
USE_AVERAGE  = False        # smallenstrans only: True -> ensemble mean instead of MEMBER_FILE

OCX_DIR          = Path("/media/jagereli/Expansion1/TLC_ISMIP7_ANT/SSA_POC/AA_SSA_ISMIP7_OCX_MAHTI")
OCX_STATES_FILE  = OCX_DIR / "ssp126_c005_states.nc"    # geometry, velocity, groundedmask, haf, cell_area...
OCX_FORCING_FILE = OCX_DIR / "ssp126_c005_forcing.nc"   # smb_total_flux (face), bmb (node)

# --- Saved intermediates (built once; reused so nothing heavy re-runs) --------
# The OCX mesh has a DIFFERENT node/face count than the SmallEnsTrans mesh, so
# each has its own basin/obs regrid — pick the file matching MEMBER_KIND.
BASINS_FILE   = "basins_mouginotGrid_ocx.nc"   if MEMBER_KIND == "ocx" else "basins_mouginotGrid.nc"
# Same satellite obs (velocity/dhdt/BedMachine) either way, just pre-interpolated
# onto whichever mesh MEMBER_KIND points at (OCX and SmallEnsTrans use different
# meshes, hence two separate files -- not "SmallEnsTrans data", just its mesh).
OBS_MESH_FILE = "obs_on_elmer_mesh_pv_ocx.nc"  if MEMBER_KIND == "ocx" else "obs_on_elmer_mesh_pv.nc"
ENS_AGG_FILE  = "ensemble_basin_aggregates3.nc"   # per-member per-catchment budget (SmallEnsTrans only)

# --- Physical constants (match COLD.lua) -------------------------------------
RHO_ICE = 917.0     # kg m-3
RHO_SW  = 1028.0    # kg m-3
GT      = 1e-12     # kg -> Gt
V_FILL_THRESHOLD = 30000.0   # |velocity| above this = fill/garbage node -> mask


### 0.2  Imports & ElmerUgrid

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import xugrid as xu
import matplotlib.pyplot as plt
import proplot as pplt              # figure styling used throughout §1/§2 plots
import itertools
from itertools import combinations as iter_combinations
# Requires in the env: xugrid, xarray, proplot, cftime (for 360_day obs calendars).


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(POSTPRO_DIR))
from ElmerUgrid import ugrid                 # registers .ugrid.to_netcdf_forpv
from ElmerUgrid.Interpolation import *
from ElmerUgrid.Projection import *


### 0.3  Observations, mesh grid & Mouginot catchments

`BASINS_FILE`/`OBS_MESH_FILE` were already resolved in §0.1 to the file matching
`MEMBER_KIND` — **the OCX mesh has a different node/face count than SmallEnsTrans**,
so basins/obs are regridded separately per mesh (both derived straight from
`ISMIPobs`, no extra external data needed).

In [ ]:
# ISMIP7 gridded observations (velocity time series, dhdt, BedMachine surface)
ISMIPobs = xr.open_dataset(OBS_ISMIP_FILE, decode_times=False)

# Mouginot catchments already regridded onto the ACTIVE mesh (face-located).
# decode_times=False: the SmallEnsTrans-tagged file carries a placeholder time
# coord in "years since 0000-01-01" units that cftime cannot parse at all
# (not just "cftime missing" -- the unit itself is unsupported for this calendar).
_basin_ds = xu.open_dataset(BASINS_FILE, decode_times=False)
basin      = _basin_ds["basins_mouginot"]          # UgridDataArray, (n_faces,)
elmer_grid = basin.ugrid.grid
print(f"Mesh ({MEMBER_KIND}):", elmer_grid.n_node, "nodes,", elmer_grid.n_face, "faces")


### 0.4  Mesh geometry, connectivity & per-face areas

Reference mesh = whichever file `MEMBER_KIND` points at (OCX states file, or the
first SmallEnsTrans member). OCX ships a real `cell_area` field — used directly;
SmallEnsTrans's `true_cell_area` is all-fill, so it's computed from node
coordinates (shoelace formula) as before.

In [ ]:
member_files = sorted(MEMBER_DIR.glob(MEMBER_GLOB))
print(f"Found {len(member_files)} SmallEnsTrans members")

MESH_REF_FILE = OCX_STATES_FILE if MEMBER_KIND == "ocx" else member_files[0]
print("Mesh reference file:", MESH_REF_FILE)


In [ ]:
# ── Open the reference mesh & pull node coordinates, connectivity, face areas ─
uds_ref = xu.open_dataset(str(MESH_REF_FILE), mask_and_scale=True, decode_times=False)
grid    = uds_ref.ugrid.grid

# x/y are (n_nodes,) in OCX, (time, n_nodes) in SmallEnsTrans (written every step
# though time-invariant for a fixed SSA mesh) -> squeeze out any time dim.
def _static_node_field(da):
    return da.values[0] if "time" in da.dims else da.values

node_xy = np.column_stack([_static_node_field(uds_ref["x"]),
                           _static_node_field(uds_ref["y"])])

fnc = grid.face_node_connectivity              # (n_faces, max_nodes)  fill=-1/0
enc = grid.edge_node_connectivity               # (n_edges, 2)
efc = grid.edge_face_connectivity               # (n_edges, 2)  -1 = boundary

valid_fnc = fnc >= 0
safe_fnc  = np.where(valid_fnc, fnc, 0)
face_xy   = np.column_stack([
    np.nanmean(np.where(valid_fnc, node_xy[safe_fnc, 0], np.nan), axis=1),
    np.nanmean(np.where(valid_fnc, node_xy[safe_fnc, 1], np.nan), axis=1),
])

basins    = basin.values
basin_ids = np.unique(basins[~np.isnan(basins)]).astype(int)
n_basins  = len(basin_ids)

if MEMBER_KIND == "ocx":
    # states.nc time = days since 1990-01-01, 360-day calendar, annual writes.
    times  = 1990.0 + uds_ref["time"].values / 360.0
    n_time = len(times)
else:
    # Confirmed via restart_time_*.nc's `elmer_time` field: 25 annual snapshots,
    # simulation years 1990-2014 (the raw `time` coord in this file is NOT the
    # absolute clock -- elmer_time is authoritative).
    times  = np.linspace(1990, 2014, 25)
    n_time = len(times)

print(f"node_xy (first 3):\n{node_xy[:3]}")
print(f"n_time={n_time}, years {times[0]:.1f}-{times[-1]:.1f}")

if MEMBER_KIND == "ocx" and "cell_area" in uds_ref:
    face_areas = uds_ref["cell_area"].values.astype(np.float64)
    print(f"cell_area (real, from XIOS): total {face_areas.sum():.3e} m2")
    uds_ref.close()
else:
    uds_ref.close()


In [ ]:
# ── SmallEnsTrans only: true_cell_area is all-fill -> compute from geometry ──
def compute_face_areas(node_xy: np.ndarray, face_node_conn: np.ndarray) -> np.ndarray:
    """Shoelace-formula face areas (m2) for arbitrary (triangle/quad) polygons."""
    n_faces, max_nodes = face_node_conn.shape
    areas = np.zeros(n_faces)
    for k in range(max_nodes):
        k_next = (k + 1) % max_nodes
        idx_k, idx_k_next = face_node_conn[:, k], face_node_conn[:, k_next]
        valid = (idx_k >= 0) & (idx_k_next >= 0)
        xk      = np.where(valid, node_xy[np.where(idx_k      >= 0, idx_k,      0), 0], 0.0)
        yk      = np.where(valid, node_xy[np.where(idx_k      >= 0, idx_k,      0), 1], 0.0)
        xk_next = np.where(valid, node_xy[np.where(idx_k_next >= 0, idx_k_next, 0), 0], 0.0)
        yk_next = np.where(valid, node_xy[np.where(idx_k_next >= 0, idx_k_next, 0), 1], 0.0)
        areas += xk * yk_next - xk_next * yk
    return np.abs(areas) / 2.0

if MEMBER_KIND != "ocx":
    face_areas = compute_face_areas(node_xy, fnc)

print(f"face_areas: min={face_areas.min():.3e}, max={face_areas.max():.3e} m2, "
      f"total={face_areas.sum():.3e} m2 (expect ~1.4e13)")


In [ ]:
# %% Cell 3 — FIXED: edge geometry (now in metres)
n0_idx = enc[:, 0]
n1_idx = enc[:, 1]

edge_vec  = node_xy[n1_idx] - node_xy[n0_idx]
edge_len  = np.linalg.norm(edge_vec, axis=1)       # now in metres, expect 100–50000 m
edge_unit = edge_vec / np.where(edge_len > 0, edge_len, 1)[:, None]

normal_left  = np.column_stack([-edge_unit[:, 1],  edge_unit[:, 0]])
normal_right = np.column_stack([ edge_unit[:, 1], -edge_unit[:, 0]])
edge_mid     = (node_xy[n0_idx] + node_xy[n1_idx]) / 2.0

print(f"edge_len: min={edge_len.min():.1f}, max={edge_len.max():.1f} m")
# expect: ~100 – 50000 m, NOT 0–360

### 0.5  Node→face helpers

In [ ]:
# Cell 4 — FIXED node_to_face  (replace the previous one)
def node_to_face(node_values: np.ndarray,
                 face_node_conn: np.ndarray) -> np.ndarray:
    """
    Average node-located values to face centres.
    node_values   : (..., n_nodes)
    face_node_conn: (n_faces, max_nodes)  fill=-1
    returns       : (..., n_faces)
    """
    valid    = face_node_conn >= 0                      # (n_faces, max_nodes)
    safe_idx = np.where(valid, face_node_conn, 0)
    gathered = node_values[..., safe_idx]               # (..., n_faces, max_nodes)
    # np.where broadcasts `valid` over all leading dims — no dimension collapse
    gathered = np.where(valid, gathered, np.nan)
    return np.nanmean(gathered, axis=-1)                # (..., n_faces)

In [ ]:
# %% Cell 4b — Helper: node-to-face min and max (for GL straddling detection)

def node_to_face_minmax(node_values: np.ndarray,
                         face_node_conn: np.ndarray):
    """
    Compute per-face min and max of node values.
    node_values   : (..., n_nodes)
    face_node_conn: (n_faces, max_nodes)  fill=-1
    Returns: face_min, face_max  each (..., n_faces)
    """
    valid    = face_node_conn >= 0
    safe_idx = np.where(valid, face_node_conn, 0)
    gathered = node_values[..., safe_idx]          # (..., n_faces, max_nodes)
    gathered = np.where(valid, gathered, np.nan)

    return np.nanmin(gathered, axis=-1), np.nanmax(gathered, axis=-1)


def get_gl_straddling_faces(gm_node: np.ndarray,
                             face_node_conn: np.ndarray) -> np.ndarray:
    """
    Identify faces that straddle the grounding line:
    at least one node grounded (gm >= 0) AND at least one floating (gm < 0).

    Parameters
    ----------
    gm_node : (..., n_nodes)  raw groundedmask node values (-1, 0, 1)

    Returns
    -------
    gl_straddle : (..., n_faces)  bool
    """
    gm_min, gm_max = node_to_face_minmax(gm_node, face_node_conn)
    # Has at least one floating node (min < 0) AND one grounded/GL node (max >= 0)
    return (gm_min < 0) & (gm_max >= 0)

### 0.6  Flux integrators (GL discharge & calving)

In [ ]:
# %% Cell 5 — Helper: GL discharge for one time step
def gl_discharge_one_step(
    gm_face_t: np.ndarray,     # (n_faces,)  grounded mask averaged to faces
    u_node_t:  np.ndarray,     # (n_nodes,)  x-velocity
    v_node_t:  np.ndarray,     # (n_nodes,)  y-velocity
    h_node_t:  np.ndarray,     # (n_nodes,)  ice thickness
    basins:    np.ndarray,     # (n_faces,)  basin IDs
    basin_ids: np.ndarray,     # (n_basins,)
) -> np.ndarray:
    """
    Compute grounding-line discharge per basin for a single time step.
 
    Grounding-line edges are edges where one adjacent face is grounded (gm > 0.5)
    and the other is floating/ocean (gm ≤ 0.5, or boundary = -1).
 
    Flux through each GL edge:
        Q = (u·n̂_out + v·n̂_out) × h_edge × L_edge    [m³ yr⁻¹ ice equiv.]
 
    n̂_out points from the grounded face toward the floating side.
    """
    # ── Identify grounding-line edges ────────────────────────────────────────
    # efc has shape (n_edges, 2): face index on each side, -1 = domain boundary.
    f0 = efc[:, 0]
    f1 = efc[:, 1]
 
    # Grounded flag for each adjacent face (-1 face → treated as floating)
    def face_grounded(fidx):
        grounded = np.zeros(len(fidx), dtype=bool)
        valid = fidx >= 0
        grounded[valid] = gm_face_t[fidx[valid]] >= 0.0
        return grounded
 
    gm_f0 = face_grounded(f0)
    gm_f1 = face_grounded(f1)
 
    # GL edge: exactly one side grounded
    is_gl = gm_f0 ^ gm_f1                             # XOR: (n_edges,)
    gl_idx = np.where(is_gl)[0]                        # indices of GL edges
 
    if len(gl_idx) == 0:
        return np.zeros(len(basin_ids))
 
    # ── Determine outward normal (from grounded toward floating) ─────────────
    # For each GL edge, identify which face is grounded
    grounded_face = np.where(gm_f0[gl_idx], f0[gl_idx], f1[gl_idx])   # (n_gl,)
 
    # Vector from edge midpoint → grounded face centroid
    to_grounded = face_xy[grounded_face] - edge_mid[gl_idx]            # (n_gl, 2)
 
    # Outward normal = the candidate normal with negative dot product toward grounded centroid
    # (i.e., the one pointing AWAY from the grounded face)
    dot_left = np.sum(normal_left[gl_idx] * to_grounded, axis=1)       # (n_gl,)
    # If dot_left < 0 → left normal already points away from grounded → use it
    n_out = np.where(
        dot_left[:, None] < 0,
        normal_left [gl_idx],
        normal_right[gl_idx],
    )                                                                    # (n_gl, 2)
 
    # ── Edge-averaged velocity and thickness ─────────────────────────────────
    ni = n0_idx[gl_idx]                                 # (n_gl,) node indices
    nj = n1_idx[gl_idx]
 
    u_edge = 0.5 * (u_node_t[ni] + u_node_t[nj])       # (n_gl,)
    v_edge = 0.5 * (v_node_t[ni] + v_node_t[nj])
    h_edge = 0.5 * (h_node_t[ni] + h_node_t[nj])
 
    # Normal velocity (positive = outward from grounded domain)
    v_norm = u_edge * n_out[:, 0] + v_edge * n_out[:, 1]               # (n_gl,)
 
    # Volumetric flux [m² yr⁻¹ × m = m³ yr⁻¹]
    flux = np.maximum(0.0, v_norm) * h_edge * edge_len[gl_idx]         # (n_gl,)
 
    # ── Assign each GL edge to a basin ───────────────────────────────────────
    # Use the basin of the grounded adjacent face
    # (boundary face → grounded_face already resolved to a valid face index)
    edge_basin = basins[grounded_face]                                  # (n_gl,)
 
    # ── Sum per basin [m³ yr⁻¹] → [Gt yr⁻¹] ─────────────────────────────────
    gl_per_basin = np.zeros(len(basin_ids))
    for i, bid in enumerate(basin_ids):
        mask = edge_basin == bid
        gl_per_basin[i] = np.nansum(flux[mask]) * RHO_ICE * GT
 
    return gl_per_basin


In [ ]:
# %% Cell OBS1 — Verify calving: edge-based cross-check

def calving_edge_flux(gm_face_t, u_node_t, v_node_t, h_node_t):
    """
    Calving flux via boundary edges adjacent to floating faces.
    Analogous to gl_discharge_one_step but for the ice front.
    Boundary edges: efc[:,0] or efc[:,1] == -1  (domain boundary)
    """
    # Boundary edges: one side has no face (-1)
    f0, f1   = efc[:, 0], efc[:, 1]
    is_bdy   = (f0 < 0) | (f1 < 0)

    # The valid adjacent face index
    valid_f  = np.where(f0 >= 0, f0, f1)             # (n_edges,)

    # Floating mask for that face
    is_float = np.zeros(len(efc), dtype=bool)
    is_float[is_bdy] = (gm_face_t[valid_f[is_bdy]] < 0.0)

    # Calving front edges: boundary AND floating
    is_cf    = is_bdy & is_float
    cf_idx   = np.where(is_cf)[0]

    if len(cf_idx) == 0:
        return 0.0, np.zeros(len(basin_ids))

    # Outward normal: points away from the ice (away from the valid face)
    to_face  = face_xy[valid_f[cf_idx]] - edge_mid[cf_idx]   # (n_cf, 2)
    dot_left = np.sum(normal_left[cf_idx] * to_face, axis=1)
    n_out    = np.where(dot_left[:, None] < 0,
                        normal_left[cf_idx], normal_right[cf_idx])

    # Edge-averaged velocity and thickness
    ni = n0_idx[cf_idx]; nj = n1_idx[cf_idx]
    u_cf = 0.5 * (u_node_t[ni] + u_node_t[nj])
    v_cf = 0.5 * (v_node_t[ni] + v_node_t[nj])
    h_cf = 0.5 * (h_node_t[ni] + h_node_t[nj])

    v_norm   = u_cf * n_out[:, 0] + v_cf * n_out[:, 1]
    flux     = np.maximum(0.0, v_norm) * h_cf * edge_len[cf_idx]  # m³/yr per edge

    # Total calving [Gt/yr]
    total_calving = flux.sum() * RHO_ICE * GT

    # Per basin (using the adjacent floating face's basin ID)
    edge_basin    = basins[valid_f[cf_idx]]
    calv_per_basin = np.zeros(len(basin_ids))
    for i, bid in enumerate(basin_ids):
        mask = edge_basin == bid
        calv_per_basin[i] = np.nansum(flux[mask]) * RHO_ICE * GT

    return total_calving, calv_per_basin


### 0.7  Per-member diagnostics (mass budget by catchment)

Two formats on disk, normalized to the same fields before the (format-agnostic)
mass-budget math:
  - **`ocx`**: geometry/velocity/`haf` from `OCX_STATES_FILE`, SMB (`smb_total_flux`,
    already **face**-located) + BMB (`bmb`, node) from `OCX_FORCING_FILE`.
  - **`smallenstrans`**: everything in one file; velocity named `"ssavelocity 1/2"`;
    SMB/BMB both node-located; no `haf` (computed from `h`/`bedrock` instead).

In [ ]:
def load_member_fields(fpath=None, kind=MEMBER_KIND):
    """Open one member (any kind) -> dict of normalized (time, n_nodes/faces)
    float32 arrays (halves peak memory vs. the netCDF's native float64 -- the
    OCX case briefly holds two large files open at once)."""
    f32 = lambda a: np.asarray(a, dtype=np.float32)
    if kind == "ocx":
        uds_s = xu.open_dataset(str(OCX_STATES_FILE),  mask_and_scale=True, decode_times=False)
        uds_f = xu.open_dataset(str(OCX_FORCING_FILE), mask_and_scale=True, decode_times=False)
        u_raw, v_raw = f32(uds_s["ssavelocity_x"].values), f32(uds_s["ssavelocity_y"].values)
        fields = dict(
            h_node    = f32(uds_s["h"].values),
            gm_node   = f32(uds_s["groundedmask"].values),
            bed_node  = f32(uds_s["bedrock"].values),
            u_node    = np.where(np.abs(u_raw) < V_FILL_THRESHOLD, u_raw, 0.0).astype(np.float32),
            v_node    = np.where(np.abs(v_raw) < V_FILL_THRESHOLD, v_raw, 0.0).astype(np.float32),
            calv_face = f32(np.nan_to_num(uds_s["calving_front_flux"].values, nan=0.0)),
            haf_node  = f32(uds_s["haf"].values) if "haf" in uds_s else None,
            bmb_node  = f32(uds_f["bmb"].values),
            smb_face  = f32(uds_f["smb_total_flux"].values),   # already face-located
            smb_node  = None,
        )
        uds_s.close(); uds_f.close()
        return fields
    else:
        fpath = fpath or member_files[0]
        uds = xu.open_dataset(str(fpath), mask_and_scale=True)
        u_raw, v_raw = f32(uds["ssavelocity 1"].values), f32(uds["ssavelocity 2"].values)
        fields = dict(
            h_node    = f32(uds["h"].values),
            gm_node   = f32(uds["groundedmask"].values),
            bed_node  = f32(uds["bedrock"].values),
            u_node    = np.where(np.abs(u_raw) < V_FILL_THRESHOLD, u_raw, 0.0).astype(np.float32),
            v_node    = np.where(np.abs(v_raw) < V_FILL_THRESHOLD, v_raw, 0.0).astype(np.float32),
            calv_face = f32(np.nan_to_num(uds["calving_front_flux"].values, nan=0.0)),
            haf_node  = None,
            bmb_node  = f32(uds["bmb"].values),
            smb_face  = None,
            smb_node  = f32(uds["smb"].values),
        )
        uds.close()
        return fields


In [ ]:
def compute_member_diagnostics(fields: dict, gl_melt: str = "fmp") -> dict:
    """
    Per-basin mass budget diagnostics for one member, from normalized `fields`
    (see load_member_fields). Math identical regardless of source format.
    Loops one YEAR at a time (not vectorized over all years) to keep peak
    memory to ~O(n_faces) rather than O(n_time * n_faces) -- the OCX case
    already holds two large files' worth of (35, ~1M) arrays in `fields`.

    BMB conventions (confirmed from Fortran SSAEffectiveBMB):
        - Fully grounded faces (all nodes grounded, gm_min >= 0): BMB = 0
        - GL-straddling faces (mixed nodes): fmp -> BMB as stored, nmp -> BMB = 0
        - Fully floating faces: BMB applied as stored
    """
    h_node, gm_node, bed_node = fields["h_node"], fields["gm_node"], fields["bed_node"]
    u_node, v_node, calv_face = fields["u_node"], fields["v_node"], fields["calv_face"]
    bmb_node = fields["bmb_node"]
    smb_face_all, smb_node = fields.get("smb_face"), fields.get("smb_node")
    haf_node = fields.get("haf_node")
    a = face_areas                                        # (n_faces,) static

    out = {k: np.zeros((n_time, n_basins)) for k in [
        "iaf_mass", "grounded_mass", "shelf_mass", "grounded_area", "floating_area",
        "smb_grounded", "smb_floating", "bmb_grounded", "bmb_floating",
        "calving_face", "calving_edge", "gl_discharge",
    ]}

    for t in range(n_time):
        h_face_t   = node_to_face(h_node[t],   fnc)
        gm_face_t  = node_to_face(gm_node[t],  fnc)
        bmb_face_t = node_to_face(bmb_node[t], fnc)
        bed_face_t = node_to_face(bed_node[t], fnc)
        smb_face_t = smb_face_all[t] if smb_face_all is not None else node_to_face(smb_node[t], fnc)

        gm_min_t, gm_max_t = node_to_face_minmax(gm_node[t], fnc)
        fully_grounded_t = gm_min_t >= 0                    # BMB = 0 here (confirmed)
        gl_straddle_t    = (gm_min_t < 0) & (gm_max_t >= 0)  # mixed nodes

        grounded_t = gm_face_t >= 0.0
        floating_t = ~grounded_t

        bmb_face_t = np.where(fully_grounded_t, 0.0, bmb_face_t)
        if gl_melt == "nmp":
            bmb_face_t = np.where(gl_straddle_t, 0.0, bmb_face_t)

        if haf_node is not None:                            # OCX: use Elmer's own haf
            haf_face_t = node_to_face(haf_node[t], fnc)
            h_af_t = np.where(grounded_t, np.maximum(0.0, haf_face_t), 0.0)
        else:                                                # SmallEnsTrans: compute it
            h_fl_t = np.maximum(0.0, -bed_face_t * (RHO_SW / RHO_ICE))
            h_af_t = np.where(grounded_t, np.maximum(0.0, h_face_t - h_fl_t), 0.0)

        iaf_per_face_t       = h_af_t                             * a * RHO_ICE * GT
        grounded_mass_face_t = np.where(grounded_t, h_face_t, 0.) * a * RHO_ICE * GT
        shelf_mass_face_t    = np.where(floating_t, h_face_t, 0.) * a * RHO_ICE * GT

        gl_t = gl_discharge_one_step(gm_face_t, u_node[t], v_node[t], h_node[t],
                                     basins, basin_ids)
        _, calv_edge_t = calving_edge_flux(gm_face_t, u_node[t], v_node[t], h_node[t])

        for i, bid in enumerate(basin_ids):
            m  = basins == bid
            mg = m & grounded_t
            mf = m & floating_t
            out["iaf_mass"]      [t, i] = np.nansum(iaf_per_face_t      [m])
            out["grounded_mass"] [t, i] = np.nansum(grounded_mass_face_t[m])
            out["shelf_mass"]    [t, i] = np.nansum(shelf_mass_face_t   [m])
            out["grounded_area"] [t, i] = np.nansum(a[mg]) * 1e-6
            out["floating_area"] [t, i] = np.nansum(a[mf]) * 1e-6
            out["smb_grounded"]  [t, i] = np.nansum(smb_face_t[mg] * a[mg]) * RHO_ICE * GT
            out["smb_floating"]  [t, i] = np.nansum(smb_face_t[mf] * a[mf]) * RHO_ICE * GT
            out["bmb_grounded"]  [t, i] = np.nansum(bmb_face_t[mg] * a[mg]) * RHO_ICE * GT
            out["bmb_floating"]  [t, i] = np.nansum(bmb_face_t[mf] * a[mf]) * RHO_ICE * GT
            out["calving_face"]  [t, i] = np.nansum(calv_face[t, mf] * a[mf]) * RHO_ICE * GT
            out["calving_edge"]  [t, i] = calv_edge_t[i]
            out["gl_discharge"]  [t, i] = gl_t[i]

    return out


## §1 — Single / averaged member vs observations

Analyses **one run** (`MEMBER_FILE`) or the **ensemble mean** (`USE_AVERAGE=True`).

### 1a — Global: discharge & mass change by catchment

`gl_discharge`, SMB and the grounded mass balance `SMB_gr − D` per Mouginot
catchment, compared to observed discharge (BMHF14) where available.

In [ ]:
# Build a per-catchment diagnostic dataset for the selected member (or the mean).
# `times`/`n_time` come from §0.4 (already format-aware: OCX 1991-2025 (35),
# SmallEnsTrans 1990-2014 (25)).
years = times

def member_diag_dataset(fpath=None, kind=MEMBER_KIND, gl_melt=MEMBER_GLMELT, label=None):
    """Load one member (any kind) -> (time, catchment) diagnostic dataset."""
    fields = load_member_fields(fpath, kind=kind)
    diag   = compute_member_diagnostics(fields, gl_melt=gl_melt)
    if label is None:
        label = "OCX C011 (Mahti)" if kind == "ocx" else Path(fpath or member_files[0]).parent.name
    ds1 = xr.Dataset(
        {k: (["time", "catchment"], v) for k, v in diag.items()},
        coords={"time": years, "catchment": basin_ids},
        attrs={"label": label},
    )
    ds1["smb"]          = ds1["smb_grounded"] + ds1["smb_floating"]
    ds1["mass_balance"] = ds1["smb_grounded"] - ds1["gl_discharge"]   # grounded MB
    return ds1

In [ ]:
if USE_AVERAGE:
    # Ensemble mean straight from the saved aggregate (SmallEnsTrans only, no re-compute).
    _ens = xr.open_dataset(ENS_AGG_FILE)
    member = _ens.mean("simu")
    member["mass_balance"] = member["smb_grounded"] - member["gl_discharge"]
    member.attrs["label"] = "ensemble mean"
    _ens.close()
    MEMBER_PATH = None
elif MEMBER_KIND == "ocx":
    MEMBER_PATH = OCX_STATES_FILE
    member = member_diag_dataset(kind="ocx")
else:
    MEMBER_PATH = MEMBER_FILE or member_files[0]   # None -> first ensemble member
    member = member_diag_dataset(MEMBER_PATH, kind="smallenstrans")

print("Analysing:", member.attrs["label"])
member

In [ ]:
# ── Observed discharge (BMHF14 / IMBIE) mapped to catchments 1..18 ───────────
# Returns (obs_years, d[time,18], derr[time,18]) or None if the file is absent.
def load_obs_discharge(path=OBS_DISCHARGE_FILE):
    if not Path(path).exists():
        print(f"[obs] {path} not on disk -> discharge obs skipped.")
        return None
    dso = xr.open_dataset(path, decode_times=False)   # raw days-since-1950, avoids cftime
    btype = dso["basin_type"].values
    imbie = btype == "imbie"
    oyears = dso["time"].values / 365.25 + 1950.0      # BMHF14 covers ~1996-2025
    sel = (oyears >= years.min() - 1) & (oyears <= years.max() + 1)
    d  = dso["discharge_mean"].values[sel][:, imbie]      # (nt, 18)
    de = dso["discharge_error"].values[sel][:, imbie]
    dso.close()
    return oyears[sel], d, de

obs_disch = load_obs_discharge()


In [ ]:
# ── Single-member vs obs: one panel per catchment ───────────────────────────
import proplot as pplt

def _slug(s: str) -> str:
    """Filesystem-safe slug for figure filenames (label -> no spaces/parens)."""
    return "".join(c if c.isalnum() else "_" for c in str(s)).strip("_")


FIGURE_DIR = Path("figures")
FIGURE_DIR.mkdir(exist_ok=True)


def savefig(fig, name: str):
    """Save a figure to FIGURE_DIR at publication-ish resolution and say so --
    §1a plots are the ones most likely to get shared with colleagues."""
    path = FIGURE_DIR / f"{name}.png"
    fig.savefig(path, dpi=200, bbox_inches="tight")
    print(f"Saved {path}")
    return path


def plot_member_by_catchment(member, variable="gl_discharge",
                             obs=obs_disch, overlay_obs=True):
    """3x6 grid, one Mouginot catchment per panel: model line (+ obs discharge)."""
    cids = [c for c in member["catchment"].values if 1 <= int(c) <= 18]
    fig, axes = pplt.subplots(nrows=3, ncols=6, figsize=(18, 9),
                              sharey=False, sharex=True, hspace=0.5, wspace=0.4)
    for ax, cid in zip(axes, cids):
        y = member[variable].sel(catchment=cid).values
        ax.plot(years, y, color="tab:blue", lw=2, label=member.attrs["label"])
        if overlay_obs and obs is not None and variable == "gl_discharge":
            oyears, d, de = obs
            ib = int(cid) - 1
            v = ~np.isnan(d[:, ib])
            ax.errorbar(oyears[v], d[v, ib], yerr=de[v, ib], fmt="o",
                        color="tab:red", ms=3, lw=1, label="BMHF14 obs", zorder=5)
        ax.set_title(f"Basin {int(cid)}", fontsize=8)
        ax.set_ylabel("Gt yr⁻¹", fontsize=7); ax.format(xminorlocator=5)
    axes[0].legend(fontsize=7, loc="upper left")
    fig.suptitle(f"{variable} — {member.attrs['label']} vs obs", fontsize=12)
    savefig(fig, f"1a_{variable}_by_catchment_{_slug(member.attrs['label'])}")
    return fig, axes

plot_member_by_catchment(member, "gl_discharge")


**Cross-check (OCX only)**: Elmer/Ice's own `ligroundf` field (grounding-line
flux, face-located, m/a) is a candidate to replace the custom edge-based
`gl_discharge_one_step` used above — cheaper and possibly more accurate, but
**not yet validated**. Compare the two totals before trusting it for anything.

In [ ]:
if MEMBER_KIND == "ocx":
    uds_f = xu.open_dataset(str(OCX_FORCING_FILE), mask_and_scale=True, decode_times=False)
    ligroundf = uds_f["ligroundf"].values          # (time, n_faces), m ice a-1
    uds_f.close()
    gl_ligroundf_total = np.nansum(np.where(ligroundf > 0, ligroundf, 0.0) *
                                   face_areas[None, :], axis=1) * RHO_ICE * GT
    gl_edge_total = member["gl_discharge"].sum("catchment").values
    fig, ax = pplt.subplots(figsize=(7, 4))
    ax.plot(years, gl_edge_total,      color="tab:blue", lw=2, label="edge method (used above)")
    ax.plot(years, gl_ligroundf_total, color="tab:orange", lw=2, ls="--", label="ligroundf (Elmer built-in)")
    ax.set_xlabel("Year"); ax.set_ylabel("Total GL discharge (Gt yr⁻¹)")
    ax.legend(loc="best", fontsize=8)
    ax.format(title="GL discharge: edge method vs ligroundf cross-check")
    savefig(fig, f"1a_gl_discharge_ligroundf_crosscheck_{_slug(member.attrs['label'])}")
else:
    print("SmallEnsTrans has no ligroundf field -> cross-check skipped.")


In [ ]:
# ── Cumulative mass change by catchment (grounded MB = SMB_gr - D), ref 2000 ──
def plot_member_cumulative_mb(member, obs=obs_disch, ref_year=2000):
    cids = [c for c in member["catchment"].values if 1 <= int(c) <= 18]
    ref  = int(np.argmin(np.abs(years - ref_year)))
    fig, axes = pplt.subplots(nrows=3, ncols=6, figsize=(18, 9),
                              sharey=False, sharex=True, hspace=0.5, wspace=0.4)
    for ax, cid in zip(axes, cids):
        mb = member["mass_balance"].sel(catchment=cid).values
        cum = np.nancumsum(mb); cum -= cum[ref]
        ax.plot(years, cum, color="tab:blue", lw=2, label=member.attrs["label"])
        if obs is not None:
            oyears, d, de = obs
            ib = int(cid) - 1; v = ~np.isnan(d[:, ib])
            if v.sum() >= 2:
                d_i = np.interp(years, oyears[v], d[v, ib], left=np.nan, right=np.nan)
                smb = member["smb_grounded"].sel(catchment=cid).values
                cum_o = np.nancumsum(smb - d_i); cum_o -= cum_o[ref]
                ax.plot(years, cum_o, color="tab:red", lw=2, ls="--",
                        label="SMB_gr − D_obs")
        ax.axhline(0, color="k", lw=0.7, ls=":")
        ax.set_title(f"Basin {int(cid)}", fontsize=8)
        ax.set_ylabel("Gt", fontsize=7); ax.format(xminorlocator=5)
    axes[0].legend(fontsize=7, loc="upper left")
    fig.suptitle(f"Cumulative mass balance (ref {ref_year}) — "
                 f"{member.attrs['label']}", fontsize=12)
    savefig(fig, f"1a_cumulative_mb_by_catchment_{_slug(member.attrs['label'])}")
    return fig, axes

plot_member_cumulative_mb(member)


### 1b — 2D maps: velocity & dH/dt vs obs

Node-located ISMIP observations pre-interpolated onto the active mesh
(`OBS_MESH_FILE`, e.g. `obs_on_elmer_mesh_pv.nc`), compared to the selected
member for a chosen catchment/year.

In [ ]:
# Load obs already interpolated onto the Elmer mesh nodes.
uds_obs = xu.open_dataset(OBS_MESH_FILE)
node_x  = node_xy[:, 0].astype(np.float32)
node_y  = node_xy[:, 1].astype(np.float32)

print(uds_obs.data_vars)


In [ ]:
def get_catchment_mask(catchment: int) -> np.ndarray:
    """Boolean node mask. (n_nodes,)"""
    return uds_obs["node_basin"].values == catchment


def get_catchment_bounds(catchment: int, pad_frac: float = 0.05) -> tuple:
    """(xlim, ylim) from node coordinates."""
    mask = get_catchment_mask(catchment)
    x_c, y_c = node_x[mask], node_y[mask]
    if len(x_c) == 0:
        raise ValueError(f"No nodes for catchment {catchment}")
    pad_x = (x_c.max() - x_c.min()) * pad_frac
    pad_y = (y_c.max() - y_c.min()) * pad_frac
    return ((x_c.min()-pad_x, x_c.max()+pad_x), (y_c.min()-pad_y, y_c.max()+pad_y))


def get_obs_velocity_at_year(year: float):
    t = int(np.argmin(np.abs(uds_obs.coords["vel_time"].values - year)))
    return uds_obs["velocity"].isel(vel_time=t), f"{uds_obs.coords['vel_time'].values[t]:.1f}"


def get_obs_dhdt_at_year(year: float):
    t = int(np.argmin(np.abs(uds_obs.coords["dhdt_time"].values - year)))
    return uds_obs["dhdt"].isel(dhdt_time=t), f"{uds_obs.coords['dhdt_time'].values[t]:.1f}"


_reprojected_cache = {}   # (fpath, kind) -> reprojected xu.Dataset, built at most once

def _reproject_mesh(fpath, kind):
    """Raw XIOS UGRID files store mesh2D_node_x/y as lon/lat degrees (not the
    separate projected x/y data vars) -- .ugrid.plot() renders straight from the
    mesh topology, so it must be reprojected or plots come out blank/misaligned.
    Same fix as the original notebook's elmer_outputs -> ismip6_init step.
    Reprojecting a ~1M-node mesh is expensive -- cache per file so functions that
    loop over years/members (plot_temporal_maps, plot_diff_maps, ...) only pay
    this cost once per file, not once per call."""
    key = (str(fpath), kind)
    if key not in _reprojected_cache:
        decode_times = False if kind == "ocx" else True
        uds_m = xu.open_dataset(str(fpath), mask_and_scale=True, decode_times=decode_times)
        uds_m.ugrid.set_crs("EPSG:4326")
        _reprojected_cache[key] = uds_m.ugrid.to_crs("EPSG:3031")
    return _reprojected_cache[key]


def load_model_snapshot_ugrid(fpath, variable: str, t_idx: int, kind: str = MEMBER_KIND):
    """One time-step node-located snapshot, any format, no node->face conversion."""
    uds_m = _reproject_mesh(fpath, kind)
    if kind == "ocx":
        if variable == "velocity":
            u = uds_m["ssavelocity_x"].isel(time=t_idx).values
            v = uds_m["ssavelocity_y"].isel(time=t_idx).values
            u = np.where(np.abs(u) < V_FILL_THRESHOLD, u, np.nan)
            v = np.where(np.abs(v) < V_FILL_THRESHOLD, v, np.nan)
            vals = np.sqrt(u**2 + v**2).astype(np.float32)
            da = uds_m["ssavelocity_x"].isel(time=t_idx).copy(data=vals)
        elif variable == "dhdt":
            da = uds_m["dhdt"].isel(time=t_idx)         # written directly by XIOS
        elif variable == "zs":
            da = uds_m["zs"].isel(time=t_idx)
    else:
        if variable == "velocity":
            u = uds_m["ssavelocity 1"].isel(time=t_idx).values
            v = uds_m["ssavelocity 2"].isel(time=t_idx).values
            u = np.where(np.abs(u) < V_FILL_THRESHOLD, u, np.nan)
            v = np.where(np.abs(v) < V_FILL_THRESHOLD, v, np.nan)
            vals = np.sqrt(u**2 + v**2).astype(np.float32)
            da = uds_m["ssavelocity 1"].isel(time=t_idx).copy(data=vals)
        elif variable == "dhdt":
            da = uds_m["h velocity"].isel(time=t_idx)   # differentiated by XIOS
        elif variable == "zs":
            da = uds_m["zs"].isel(time=t_idx)

    return da                          # (n_nodes,) UgridDataArray


def _plot_catch(ax, da, catchment: int, xlim, ylim, cmap, vmin, vmax):
    """Mask to catchment, clip to its bounding box, and plot with xugrid's own
    plotting -- no hand-rolled Triangulation/tripcolor needed.

    NOTE: earlier versions hand-rolled a matplotlib Triangulation + tripcolor
    call here to work around what looked like an xugrid plotting bug
    (`.ugrid.plot()` raised `TypeError: tripcolor() takes 4 positional
    arguments but 5 were given` for node-located fields). Isolated testing
    showed `.ugrid.plot()` works fine on a *plain* matplotlib Axes -- the bug
    is specific to proplot 0.9.7's axes wrapper mis-forwarding xugrid's
    Gouraud-shaded tripcolor call -- so these map panels use plain
    `plt.subplots()` (not `pplt.subplots()`).
    Calling `.ugrid.plot()` on the full ~1-2M-face continental mesh (even
    NaN-masked outside the catchment) rebuilds the whole triangulation every
    time: ~10s/call, no caching, so a 4-year x 3-row figure would take minutes.
    `.ugrid.clip_box()` first cuts that to ~1s/call -- it builds a spatial
    index once per underlying grid (~10-15s the very first call) and reuses it
    for every later clip against that same grid, which is exactly the
    plot_temporal_maps access pattern (same file, many years).
    """
    catch_mask = (uds_obs["node_basin"] == catchment).values
    da_masked = da.copy(data=np.where(catch_mask, da.values, np.nan).astype(np.float32))
    da_clip = da_masked.ugrid.clip_box(xlim[0], ylim[0], xlim[1], ylim[1])
    p = da_clip.ugrid.plot(ax=ax, cmap=cmap, vmin=vmin, vmax=vmax, add_colorbar=False)
    ax.set_xlim(xlim); ax.set_ylim(ylim)
    ax.set_aspect("equal"); ax.axis("off"); ax.set_title("")
    return p

print("2D-map helpers ready (format-aware).")


In [ ]:
# ── Single-member 3-panel map: model | obs | (model − obs) ───────────────────
def plot_member_vs_obs_2d(fpath, catchment, year, variable="velocity", kind=MEMBER_KIND,
                          cmap_abs="viridis", cmap_diff="RdBu_r",
                          vmax_abs=None, vmax_diff=None):
    xlim, ylim = get_catchment_bounds(catchment)
    t_idx = int(np.argmin(np.abs(times - year)))
    mod_da = load_model_snapshot_ugrid(fpath, variable, t_idx, kind=kind)
    if variable == "velocity":
        obs_da, obs_date = get_obs_velocity_at_year(year)
    else:
        obs_da, obs_date = get_obs_dhdt_at_year(year)
    diff_da = mod_da - obs_da

    m = get_catchment_mask(catchment)
    if vmax_abs is None:
        a = np.concatenate([mod_da.values[m], obs_da.values[m]])
        vmax_abs = np.nanpercentile(a[np.isfinite(a)], 98)
    if vmax_diff is None:
        d = diff_da.values[m]
        vmax_diff = np.nanpercentile(np.abs(d[np.isfinite(d)]), 95)
    vmin_abs = 0.0 if variable == "velocity" else -vmax_abs

    fig, axes = plt.subplots(ncols=3, figsize=(15, 5))
    p0 = _plot_catch(axes[0], mod_da,  catchment, xlim, ylim, cmap_abs, vmin_abs, vmax_abs)
    p1 = _plot_catch(axes[1], obs_da,  catchment, xlim, ylim, cmap_abs, vmin_abs, vmax_abs)
    p2 = _plot_catch(axes[2], diff_da, catchment, xlim, ylim, cmap_diff, -vmax_diff, vmax_diff)
    axes[0].set_title(f"model {int(times[t_idx])}", fontsize=9)
    axes[1].set_title(f"obs {obs_date}", fontsize=9)
    axes[2].set_title("model − obs", fontsize=9)
    fig.colorbar(p1, ax=[axes[0], axes[1]], location="bottom", shrink=0.6, label=variable)
    fig.colorbar(p2, ax=axes[2], location="bottom", shrink=0.6, label="model − obs")
    fig.suptitle(f"{variable} — catchment {catchment}", fontsize=12)
    return fig, axes

plot_member_vs_obs_2d(MEMBER_PATH, catchment=11, year=2005, variable="velocity")
plot_member_vs_obs_2d(MEMBER_PATH, catchment=11, year=2005, variable="dhdt")


In [ ]:
def plot_temporal_maps(
    catchment: int, fpath, kind: str = MEMBER_KIND, variable: str = "velocity",
    years_to_plot: list = (2001, 2005, 2010, 2014), show_obs: bool = True,
    cmap_abs: str = "viridis", cmap_diff: str = "RdBu_r",
    vmin_abs: float = None, vmax_abs: float = None, vmax_diff: float = None,
):
    """Rows: model | obs | (model-obs); one column per year.

    Titles include each panel's catchment-mean value. The colour scale is
    shared across columns (needed for a fair year-to-year comparison), which
    means real but modest changes -- a few percent over a decade is typical
    for most catchments -- can be hard to see by eye, especially where a
    fast-flowing core saturates the colormap. The mean annotation makes the
    real (if visually subtle) change legible even when the maps look similar.
    """
    xlim, ylim = get_catchment_bounds(catchment)
    catch_mask = get_catchment_mask(catchment)
    n_years = len(years_to_plot)
    n_rows  = 3 if show_obs else 1

    def _mod_obs(year):
        t_idx = int(np.argmin(np.abs(times - year)))
        mod_da = load_model_snapshot_ugrid(fpath, variable, t_idx, kind=kind)
        obs_da, obs_date = (get_obs_velocity_at_year(year) if variable == "velocity"
                            else get_obs_dhdt_at_year(year))
        return t_idx, mod_da, obs_da, obs_date

    if any(v is None for v in [vmin_abs, vmax_abs, vmax_diff]):
        all_abs, all_diff = [], []
        for year in years_to_plot:
            _, mod_da, obs_da, _ = _mod_obs(year)
            mod, obs = mod_da.values, obs_da.values
            all_abs.extend([mod[catch_mask & np.isfinite(mod)], obs[catch_mask & np.isfinite(obs)]])
            d = (mod - obs)[catch_mask & np.isfinite(mod - obs)]
            all_diff.append(d)
        flat_abs  = np.concatenate(all_abs)
        flat_diff = np.concatenate(all_diff)
        if vmin_abs  is None: vmin_abs  = 0.0 if variable == "velocity" else np.nanpercentile(flat_abs, 2)
        if vmax_abs  is None: vmax_abs  = np.nanpercentile(flat_abs, 98)
        if vmax_diff is None: vmax_diff = np.nanpercentile(np.abs(flat_diff[np.isfinite(flat_diff)]), 95)

    fig, axes = plt.subplots(nrows=n_rows, ncols=n_years, figsize=(4*n_years, 4*n_rows))
    if n_rows == 1:
        axes = axes[None, :]

    p_abs = p_diff = None
    for col, year in enumerate(years_to_plot):
        t_idx, mod_da, obs_da, obs_date = _mod_obs(year)
        mod_mean = np.nanmean(np.where(catch_mask, mod_da.values, np.nan))
        p_abs = _plot_catch(axes[0, col], mod_da, catchment, xlim, ylim, cmap_abs, vmin_abs, vmax_abs)
        axes[0, col].set_title(f"model {int(times[t_idx])}\nmean={mod_mean:.0f}", fontsize=8)
        if show_obs:
            obs_mean = np.nanmean(np.where(catch_mask, obs_da.values, np.nan))
            p_abs = _plot_catch(axes[1, col], obs_da, catchment, xlim, ylim, cmap_abs, vmin_abs, vmax_abs)
            axes[1, col].set_title(f"obs {obs_date}\nmean={obs_mean:.0f}", fontsize=8)
            diff_da = mod_da - obs_da
            p_diff = _plot_catch(axes[2, col], diff_da, catchment, xlim, ylim, cmap_diff, -vmax_diff, vmax_diff)
            axes[2, col].set_title("model − obs", fontsize=8)
        for r in range(n_rows):
            axes[r, col].set_xlim(xlim); axes[r, col].set_ylim(ylim)
            axes[r, col].set_aspect("equal"); axes[r, col].axis("off")

    if p_abs  is not None:
        rows = [axes[0, c] for c in range(n_years)] + ([axes[1, c] for c in range(n_years)] if show_obs else [])
        fig.colorbar(p_abs, ax=rows, location="left", shrink=0.6, label=variable)
    if p_diff is not None:
        fig.colorbar(p_diff, ax=[axes[2, c] for c in range(n_years)], location="left", shrink=0.6, label="model − obs")
    fig.suptitle(f"{variable} — catchment {catchment} — {'OCX C011' if kind=='ocx' else Path(fpath).parent.name}",
                fontsize=12)
    return fig, axes

plot_temporal_maps(catchment=11, fpath=MEMBER_PATH, variable="velocity",
                   years_to_plot=[2001, 2005, 2010, 2014] if MEMBER_KIND != "ocx"
                                 else [1995, 2005, 2015, 2025])


## §2 — Ensemble by characteristics

Compare members grouped by their design parameters, quantify parameter
sensitivity (Sobol), and show the ensemble envelope against observations.

### 2a — Member metadata & per-member aggregate

`meta` parses each member's design parameters from its folder name (SmallEnsTrans
convention — ensemble-only, so it lives here, not in §0).

`ensemble_basin_aggregates3.nc` holds `(time, simu, catchment)` diagnostics for
every member. It already exists — **run the build cell only to regenerate it**
(loops all members, slow).

In [ ]:
# Cell 2b — Parse member metadata from folder names  (FIXED)
import re
import pandas as pd

FOLDER_RE = re.compile(
    r"TRANSIENT(\d+)(nemo|sat)(fmp|nmp)(budd|rc)(cst|jag)(c_cut|c)$"
)

def parse_folder(folder_name: str) -> dict:
    m = FOLDER_RE.match(folder_name)
    if m is None:
        raise ValueError(f"Could not parse folder name: {folder_name!r}")
    inv, ocean, gl_melt, fric_law, n_hyp, c_ext = m.groups()
    return {
        "inv":      int(inv),        # inversion member (3, 38 or 39), NOT the transient simu index
        "ocean":    ocean,
        "gl_melt":  gl_melt,
        "fric_law": fric_law,
        "n_hyp":    n_hyp,
        "c_ext":    c_ext,
        "folder":   folder_name,
    }

# f.parent.name gives the folder (e.g. TRANSIENT38nemofmpbuddcstc)
# f.name gives the .nc file inside (e.g. restart_aa_trans_41.nc)
meta = pd.DataFrame([parse_folder(f.parent.name) for f in member_files])
meta.index.name = "simu_idx"
display(meta)

In [ ]:
# LOAD the saved aggregate (fast). Skip the build cell below unless regenerating.
ds = xr.open_dataset(ENS_AGG_FILE)
years = ds["time"].values
ds["smb"] = ds["smb_floating"] + ds["smb_grounded"]
ds["bmb"] = ds["bmb_floating"] + ds["bmb_grounded"]
print(ds.sizes)


In [ ]:
# === REGENERATE the aggregate (slow: loops every member). Normally skip. ===

# %% Cell 7 — UPDATED main loop: pass gl_melt per member
n_members = len(member_files)

keys = [
    "iaf_mass", "grounded_mass", "shelf_mass",
    "grounded_area", "floating_area",
    "smb_grounded", "smb_floating",
    "bmb_grounded", "bmb_floating",
    "calving_face", "calving_edge",
    "gl_discharge",
]
all_data = {k: np.full((n_time, n_members, n_basins), np.nan) for k in keys}

for m, (fpath, gl_melt_m) in enumerate(zip(member_files,
                                             meta["gl_melt"].values)):
    print(f"[{m+1:3d}/{n_members}]  {fpath.name}  (gl_melt={gl_melt_m})",
          flush=True)
    uds  = xu.open_dataset(str(fpath), mask_and_scale=True)
    diag = compute_member_diagnostics(uds, gl_melt=gl_melt_m)
    uds.close()
    for k in keys:
        all_data[k][:, m, :] = diag[k]

print("Done.")

# %% Cell 8 — UPDATED output netCDF with both calving methods

years = np.linspace(1990, 2014, 25)

def make_rate(arr): return np.gradient(arr, 1.0, axis=0)

attrs = {
    "iaf_mass":      ("Ice above flotation mass",                    "Gt"),
    "grounded_mass": ("Total grounded ice mass",                     "Gt"),
    "shelf_mass":    ("Total ice shelf mass",                        "Gt"),
    "grounded_area": ("Grounded ice area",                           "km2"),
    "floating_area": ("Floating ice area",                           "km2"),
    "smb_grounded":  ("SMB over grounded area",                      "Gt yr-1"),
    "smb_floating":  ("SMB over floating area",                      "Gt yr-1"),
    "bmb_grounded":  ("BMB over grounded area",                      "Gt yr-1"),
    "bmb_floating":  ("BMB over floating area",                      "Gt yr-1"),
    "calving_face":  ("Calving flux — face method (calving_front_flux × area)", "Gt yr-1"),
    "calving_edge":  ("Calving flux — edge method (v·n × h × L)",    "Gt yr-1"),
    "gl_discharge":  ("Ice discharge at grounding line",             "Gt yr-1"),
    "iaf_rate":      ("IAF mass change rate",                        "Gt yr-1"),
    "grounded_rate": ("Grounded mass change rate",                   "Gt yr-1"),
    "shelf_rate":    ("Ice shelf mass change rate",                  "Gt yr-1"),
}

dvars = {}
for k in keys:
    long, unit = attrs[k]
    dvars[k] = xr.DataArray(
        all_data[k], dims=["time", "simu", "catchment"],
        attrs={"long_name": long, "units": unit},
    )

for src, rk in [("iaf_mass",      "iaf_rate"),
                ("grounded_mass", "grounded_rate"),
                ("shelf_mass",    "shelf_rate")]:
    long, unit = attrs[rk]
    dvars[rk] = xr.DataArray(
        make_rate(all_data[src]), dims=["time", "simu", "catchment"],
        attrs={"long_name": long, "units": unit},
    )

ds_out = xr.Dataset(
    dvars,
    coords={
        "time":      xr.DataArray(years, dims=["time"],
                                  attrs={"long_name": "Year", "units": "yr"}),
        "simu":      xr.DataArray(np.arange(n_members), dims=["simu"]),
        "catchment": xr.DataArray(basin_ids, dims=["catchment"],
                                  attrs={"long_name": "Mouginot basin ID"}),
        "folder":    ("simu", meta["folder"].values),
        "ocean":     ("simu", meta["ocean"].values),
        "gl_melt":   ("simu", meta["gl_melt"].values),
        "fric_law":  ("simu", meta["fric_law"].values),
        "n_hyp":     ("simu", meta["n_hyp"].values),
        "c_ext":     ("simu", meta["c_ext"].values),
        "inv":       ("simu", meta["inv"].values),
    },
    attrs={
        "title":        "Elmer/Ice ensemble — Mouginot basin diagnostics",
        "rho_ice":      f"{RHO_ICE} kg m-3",
        "rho_sw":       f"{RHO_SW} kg m-3",
        "area_source":  "Computed from mesh geometry (shoelace formula)",
        "calving_note": ("Two calving methods saved: "
                         "calving_face = calving_front_flux*area (XIOS variable); "
                         "calving_edge = edge-flux integral at floating boundary"),
    },
)

ds_out.to_netcdf("ensemble_basin_aggregates3.nc", mode="w")
print(ds_out)

### 2b — Parameter labels & effect plots

In [ ]:
# Parameter / variable labels & colour cycle (ds already loaded in 2a).
PARAMS = ["inv", "ocean", "gl_melt", "fric_law", "n_hyp", "c_ext"]

PARAM_LABELS = {
    "inv":      "Inversion member",
    "ocean":    "Ocean forcing",
    "gl_melt":  "GL melt parametrisation",
    "fric_law": "Friction law",
    "n_hyp":    "Effective pressure",
    "c_ext":    "Friction extension",
}
VAR_LABELS = {
    "iaf_mass":     ("IAF mass",             "Gt"),
    "iaf_rate":     ("IAF mass change rate", "Gt yr⁻¹"),
    "smb":          ("SMB",                  "Gt yr⁻¹"),
    "bmb":          ("BMB",                  "Gt yr⁻¹"),
    "calving":      ("calving",              "Gt yr⁻¹"),
    "gl_discharge": ("GL discharge",         "Gt yr⁻¹"),
}
CYCLE = pplt.Cycle("tab10")


In [ ]:
# %% Cell P2 — Core plotting function
def plot_parameter_effect(
    catchment,
    params,
    variable: str = "iaf_mass",
    ax=None,
    title = None,
):
    """
    Plot sub-ensemble median ± min/max for a given catchment and parameter grouping.
 
    Parameters
    ----------
    catchment : int or 'all'
        Mouginot basin ID, or 'all' to sum over all catchments.
    params : str or list of str
        One or more of: 'inv', 'ocean', 'gl_melt', 'fric_law', 'n_hyp', 'c_ext'.
        All unique combinations of the chosen parameters define the sub-ensembles.
    variable : str
        One of 'iaf_mass', 'iaf_rate', 'smb', 'gl_discharge'.
    ax : proplot Axes, optional
        If None a new figure is created.
    title : str, optional
        Override the auto-generated title.
 
    Returns
    -------
    fig, ax  (or just ax if an existing axes was passed in)
    """
    if isinstance(params, str):
        params = [params]
 
    # ── Select variable & catchment ──────────────────────────────────────────
    da = ds[variable]                              # (time, simu, catchment)
 
    if catchment == "all":
        data = da.sum("catchment")                 # (time, simu)
        catch_label = "all catchments"
    else:
        data = da.sel(catchment=catchment)         # (time, simu)
        catch_label = f"catchment {catchment}"
 
    # data.values : (n_time, n_simu)
    data_np = data.values
 
    # ── Build sub-ensemble groups ────────────────────────────────────────────
    # Retrieve the coordinate arrays for the chosen params (length = n_simu)
    coord_arrays = {p: ds[p].values for p in params}
 
    # All unique combinations
    unique_vals   = [np.unique(coord_arrays[p]) for p in params]
    combinations  = list(itertools.product(*unique_vals))
 
    # ── Create figure if needed ───────────────────────────────────────────────
    standalone = ax is None
    if standalone:
        fig, ax = pplt.subplots(figsize=(7, 4))
    else:
        fig = ax.figure
 
    colors = [c["color"] for c in itertools.islice(CYCLE, len(combinations))]
 
    for combo, color in zip(combinations, colors):
        # Boolean mask over simu dimension
        mask = np.ones(ds.sizes["simu"], dtype=bool)
        for p, v in zip(params, combo):
            mask &= (coord_arrays[p] == v)
 
        if mask.sum() == 0:
            continue                               # combination not present
 
        sub = data_np[:, mask]                     # (n_time, n_members_in_group)
 
        median = np.nanmedian(sub, axis=1)
        lo     = np.nanmin   (sub, axis=1)
        hi     = np.nanmax   (sub, axis=1)
 
        label = " / ".join(str(v) for v in combo)
 
        ax.plot(years, median, color=color, label=label, lw=1.8)
        ax.fill_between(years, lo, hi, color=color, alpha=0.18)
 
    # ── Formatting ───────────────────────────────────────────────────────────
    var_long, var_unit = VAR_LABELS[variable]
    param_str = " × ".join(PARAM_LABELS.get(p, p) for p in params)
 
    ax.set_xlabel("Year")
    ax.set_ylabel(f"{var_long} ({var_unit})")
    ax.legend(loc="best", ncols=1, fontsize=7, title=param_str, titlefontsize=8)
    ax.format(xminorlocator=1)
 
    if standalone:
        fig.savefig(
            f"{variable}_catch{catchment}_{'_x_'.join(params)}.png",
            dpi=150, bbox_inches="tight",
        )
        return fig, ax
    return ax


In [ ]:
# %% Cell P2b — Core plotting function with observations
def plot_parameter_effect_obs(
    catchment,
    params,
    variable: str = "iaf_mass",
    ax=None,
    title = None,
    ref_year: int = 2000,
):
    """
    Like plot_parameter_effect, but overlays observations as a black dashed line.

    Obs definition by variable:
      iaf_mass     : cum. integral of (SMB_gr - D_obs), anchored to the model
                     ensemble-median IAF mass at ref_year (so both are total mass)
      iaf_rate     : annual SMB_gr - D_obs
      gl_discharge : D_obs (IMBIE) directly
      smb          : no obs
    """
    if isinstance(params, str):
        params = [params]

    # ── Select variable & catchment ──────────────────────────────────────────
    da = ds[variable]
    if catchment == "all":
        data = da.sum("catchment")
        catch_label = "all catchments"
        basin_indices = [int(cid) - 1 for cid in obs_catchment_ids]
    else:
        data = da.sel(catchment=catchment)
        catch_label = f"catchment {catchment}"
        basin_indices = [int(catchment) - 1]

    data_np = data.values   # (n_time, n_simu)

    # ── Helper: observed discharge interpolated & summed over basins ──────────
    def _obs_discharge():
        d_tot = np.zeros(len(years))
        any_valid = False
        for i_basin in basin_indices:
            d_obs_b = d_imbie[:, i_basin]
            valid   = ~np.isnan(d_obs_b)
            if valid.sum() < 2:
                continue
            any_valid = True
            d_tot += np.interp(
                years, obs_years_sel[valid], d_obs_b[valid],
                left=np.nan, right=np.nan,
            )
        return d_tot if any_valid else None

    # ── Build observed series ─────────────────────────────────────────────────
    obs_plot  = None
    obs_label = None

    if variable in ("iaf_mass", "iaf_rate"):
        smb_gr = ds["smb_grounded"]
        if catchment == "all":
            smb_np = smb_gr.sum("catchment").values.mean(axis=1)
        else:
            smb_np = smb_gr.sel(catchment=catchment).values.mean(axis=1)

        d_obs_total = _obs_discharge()
        if d_obs_total is not None:
            mb_obs_annual = smb_np - d_obs_total            # Gt yr⁻¹

            if variable == "iaf_mass":
                ref_idx = np.where(years == ref_year)[0][0]
                cum = np.nancumsum(mb_obs_annual)
                cum -= cum[ref_idx]                         # change relative to ref
                # anchor to model total mass at ref_year → both on total-mass scale
                model_ref = np.nanmedian(data_np[ref_idx, :])
                obs_plot  = model_ref + cum
                obs_label = f"SMB_gr − D_obs (anchored {ref_year})"
            else:  # iaf_rate
                obs_plot  = mb_obs_annual
                obs_label = "SMB_gr − D_obs"

    elif variable == "gl_discharge":
        obs_plot  = _obs_discharge()
        obs_label = "D_obs (IMBIE)"
    # variable == "smb": no obs counterpart, obs_plot stays None

    # ── Create figure if needed ───────────────────────────────────────────────
    standalone = ax is None
    if standalone:
        fig, ax = pplt.subplots(figsize=(7, 4))
    else:
        fig = ax.figure

    # ── Sub-ensemble groups ───────────────────────────────────────────────────
    coord_arrays = {p: ds[p].values for p in params}
    unique_vals  = [np.unique(coord_arrays[p]) for p in params]
    combinations = list(itertools.product(*unique_vals))
    colors = [c["color"] for c in itertools.islice(CYCLE, len(combinations))]

    for combo, color in zip(combinations, colors):
        mask = np.ones(ds.sizes["simu"], dtype=bool)
        for p, v in zip(params, combo):
            mask &= (coord_arrays[p] == v)
        if mask.sum() == 0:
            continue

        sub    = data_np[:, mask]
        median = np.nanmedian(sub, axis=1)
        lo     = np.nanmin   (sub, axis=1)
        hi     = np.nanmax   (sub, axis=1)
        label  = " / ".join(str(v) for v in combo)

        ax.plot(years, median, color=color, label=label, lw=1.8)
        ax.fill_between(years, lo, hi, color=color, alpha=0.18)

    # ── Observations overlay ──────────────────────────────────────────────────
    if obs_plot is not None:
        ax.plot(years, obs_plot, color="k", lw=1.8, linestyle="--",
                label=obs_label, zorder=10)

    # ── Formatting ───────────────────────────────────────────────────────────
    var_long, var_unit = VAR_LABELS[variable]
    param_str = " × ".join(PARAM_LABELS.get(p, p) for p in params)

    ax.set_xlabel("Year")
    ax.set_ylabel(f"{var_long} ({var_unit})")
    ax.legend(loc="best", ncols=1, fontsize=7, title=param_str, titlefontsize=8)
    ax.format(xminorlocator=1)

    if title is not None:
        ax.set_title(title, fontsize=9)

    if standalone:
        fig.savefig(
            f"{variable}_obs_catch{catchment}_{'_x_'.join(params)}.png",
            dpi=150, bbox_inches="tight",
        )
        return fig, ax
    return ax

In [ ]:
# %% Cell P3 — Convenience wrapper: panel figure for multiple variables
def plot_multi_variable(
    catchment,
    params,
    variables: list = ("iaf_mass", "iaf_rate", "smb", "gl_discharge"),
    ncols: int = 2,
):
    """
    Same as plot_parameter_effect but tiled over several variables.
    """
    if isinstance(params, str):
        params = [params]
 
    nrows = int(np.ceil(len(variables) / ncols))
    fig, axes = pplt.subplots(nrows=nrows, ncols=ncols,
                               figsize=(6 * ncols, 4 * nrows),
                               sharey=False)
 
    for ax, var in zip(axes, variables):
        plot_parameter_effect(catchment, params, variable=var, ax=ax)
 
    # Hide unused panels
    for ax in axes[len(variables):]:
        ax.set_visible(False)
 
    param_str = " × ".join(PARAM_LABELS.get(p, p) for p in params)
    if catchment == "all":
        catch_label = "all catchments"
    else:
        catch_label = f"catchment {catchment}"
 
    fig.suptitle(f"{catch_label}  —  grouped by {param_str}", fontsize=11, y=1.01)
    fig.savefig(
        f"multi_var_catch{catchment}_{'_x_'.join(params)}.png",
        dpi=150, bbox_inches="tight",
    )
    return fig, axes


In [ ]:
# %% Cell P5 — Parameter matrix plot
def plot_parameter_matrix(
    catchment,
    variable: str = "iaf_mass",
    params: list = PARAMS,
):
    """
    Matrix of sub-ensemble plots.
    Diagonal    : single parameter grouping
    Lower left  : two-parameter combination
    Upper right : empty
    """
    n = len(params)
    fig, axes = pplt.subplots(
        nrows=n, ncols=n,
        figsize=(3.5 * n, 3 * n),
        sharey=False, sharex=True,
        hspace=0.5, wspace=0.4,
    )

    var_long, var_unit = VAR_LABELS[variable]
    catch_label = "all catchments" if catchment == "all" else f"catchment {catchment}"

    for i in range(n):
        for j in range(n):
            ax = axes[i, j]

            if j > i:
                # Upper triangle — hide
                ax.set_visible(False)

            elif i == j:
                # Diagonal — single parameter
                plot_parameter_effect(
                    catchment, params[i], variable=variable, ax=ax,
                    title=PARAM_LABELS.get(params[i], params[i]),
                )

            else:
                # Lower triangle — two parameters
                plot_parameter_effect(
                    catchment, [params[j], params[i]], variable=variable, ax=ax,
                    title=f"{PARAM_LABELS.get(params[j], params[j])}\n× {PARAM_LABELS.get(params[i], params[i])}",
                )

            # Only show y-label on leftmost column
            if j > 0 and ax.get_visible():
                ax.set_ylabel("")

            # Only show x-label on bottom row
            if i < n - 1 and ax.get_visible():
                ax.set_xlabel("")

    fig.suptitle(
        f"{var_long} — {catch_label}",
        fontsize=13, y=1.01,
    )
    fig.savefig(
        f"matrix_{variable}_catch{catchment}.png",
        dpi=150, bbox_inches="tight",
    )
    return fig, axes



In [ ]:
# ── Example parameter-effect calls ──────────────────────────────────────────
plot_parameter_effect("all", "ocean",   variable="gl_discharge")
plot_parameter_effect(11,    ["ocean", "gl_melt"], variable="iaf_mass")
plot_multi_variable("all", "fric_law")


### 2c — Ensemble vs observations

The observed-discharge overlay (`plot_parameter_effect_obs`) needs the BMHF14
arrays. Build them here (reusing the Part-1 loader) so the ensemble envelope can
be drawn against obs.

In [ ]:
# Expose obs discharge in the shape plot_parameter_effect_obs expects.
if obs_disch is not None:
    obs_years_sel, d_imbie, de_imbie = obs_disch
    obs_catchment_ids = np.arange(1, 19)
    plot_parameter_effect_obs("all", "ocean", variable="gl_discharge")
else:
    print("BMHF14 discharge obs not available -> obs overlay skipped.")


### 2d — Sobol sensitivity indices (functional ANOVA)

In [ ]:
# %% Cell S1 — Core Sobol computation (pure functional ANOVA)
from itertools import combinations as iter_combinations
import pandas as pd
 
def _closed_variance(data_t, pvals, subset):
    """
    V[E[Y | X_subset]]  —  variance of conditional means for a parameter subset.
    Handles unbalanced group sizes (inv has 3 levels, others have 2).
 
    Parameters
    ----------
    data_t : (n_simu,) float array — output values at one time step
    pvals  : dict  param_name → (n_simu,) array of parameter values
    subset : list of param names
    """
    n          = len(data_t)
    grand_mean = np.mean(data_t)
 
    combos = list(itertools.product(*[np.unique(pvals[p]) for p in subset]))
    V = 0.0
    for combo in combos:
        mask = np.ones(n, dtype=bool)
        for p, v in zip(subset, combo):
            mask &= (pvals[p] == v)
        if mask.sum() == 0:
            continue
        p_k   = mask.sum() / n          # group weight (handles unbalanced design)
        mu_k  = np.mean(data_t[mask])
        V    += p_k * mu_k ** 2
 
    return V - grand_mean ** 2
 
 
def compute_sobol(data_t, pvals, max_order=3):
    """
    Compute all Sobol indices up to `max_order` for one time step.
 
    Parameters
    ----------
    data_t    : (n_simu,) array
    pvals     : dict  param_name → (n_simu,) array
    max_order : int, 1–3
 
    Returns
    -------
    S : dict  tuple_of_param_names → Sobol index (float)
        e.g. {('inv',): 0.12,  ('inv','ocean'): 0.03, ...}
    total_var : float
    """
    params    = list(pvals.keys())
    total_var = np.var(data_t)
 
    if total_var < 1e-30:
        return {}, 0.0
 
    # Step 1: closed variances V[E[Y|X_u]] for every subset up to max_order
    V_closed = {}
    for order in range(1, max_order + 1):
        for subset in iter_combinations(params, order):
            V_closed[subset] = _closed_variance(data_t, pvals, list(subset))
 
    # Step 2: pure interaction variances via inclusion-exclusion
    V_pure = {}
    for order in range(1, max_order + 1):
        for subset in iter_combinations(params, order):
            V_pure[subset] = V_closed[subset]
            for sub_order in range(1, order):
                for sub in iter_combinations(subset, sub_order):
                    V_pure[subset] -= V_pure[sub]
 
    S = {subset: max(0.0, V_pure[subset]) / total_var for subset in V_pure}
    return S, total_var


In [ ]:
# %% Cell S2 — Compute Sobol indices for all time steps
def build_sobol_timeseries(
    catchment,
    variable: str  = "iaf_mass",
    params:   list = PARAMS,
    max_order: int = 3,
):
    """
    Returns a DataFrame with columns = parameter subsets (as strings),
    rows = time steps.
    """
    da   = ds[variable]
    data = da.sum("catchment") if catchment == "all" else da.sel(catchment=catchment)
    data_np = data.values                              # (n_time, n_simu)
 
    pvals = {p: ds[p].values for p in params}
 
    records = []
    for t in range(len(years)):
        S, _ = compute_sobol(data_np[t], pvals, max_order=max_order)
        row = {" × ".join(k): v for k, v in S.items()}
        records.append(row)
 
    df = pd.DataFrame(records, index=years)
    df.index.name = "year"
    return df


In [ ]:
# %% Cell S3 — Plotting function
def plot_sobol_timeseries(
    sobol_df: pd.DataFrame,
    catchment,
    variable: str = "iaf_mass",
    top_n_interactions: int = 6,
    stacked: bool = True,
):
    """
    Three-panel figure: 1st order (stacked area or lines),
                        2nd order (top_n lines),
                        3rd order (top_n lines).
 
    Parameters
    ----------
    stacked            : if True, 1st-order panel is a stacked area (shows
                         how much total variance is explained).
    top_n_interactions : how many 2nd/3rd order indices to label individually;
                         the rest are grouped as 'others'.
    """
    s1 = sobol_df[[c for c in sobol_df if c.count("×") == 0]]
    s2 = sobol_df[[c for c in sobol_df if c.count("×") == 1]]
    s3 = sobol_df[[c for c in sobol_df if c.count("×") == 2]]
 
    var_long, var_unit = VAR_LABELS[variable]
    catch_label = "all catchments" if catchment == "all" else f"catchment {catchment}"
 
    fig, axes = pplt.subplots(nrows=3, ncols=1,
                               figsize=(8, 10),
                               sharey=False, hspace=0.6)
    ax1, ax2, ax3 = axes
 
    # ── Friendly labels for single parameters ────────────────────────────────
    def friendly(col):
        parts = [p.strip() for p in col.split("×")]
        return " × ".join(PARAM_LABELS.get(p, p) for p in parts)
 
    colors10 = [c["color"] for c in itertools.islice(CYCLE, max(len(s1.columns), top_n_interactions))]
 
    # ── Panel 1: 1st-order Sobol indices ────────────────────────────────────
    labels1 = [friendly(c) for c in s1.columns]
 
    if stacked:
        ax1.stackplot(
            years,
            [s1[c].values for c in s1.columns],
            labels=labels1,
            colors=colors10[:len(s1.columns)],
            alpha=0.85,
        )
        # Overlay total explained (sum of all orders) as dashed line
        total_explained = sobol_df.sum(axis=1)
        ax1.plot(years, total_explained.values, color="k", lw=1.2,
                 linestyle="--", label="Total explained", zorder=5)
    else:
        for col, label, color in zip(s1.columns, labels1, colors10):
            ax1.plot(years, s1[col].values, label=label, color=color, lw=2)
 
    ax1.set_ylabel("Sobol index  S_i")
    ax1.set_title("1st order — main effects")
    ax1.legend(loc="upper left", ncols=2, fontsize=8)
    ax1.format(yformatter="%.2f", xminorlocator=1, ylim=(0, None))
 
    # ── Panel 2: 2nd-order (top N by mean value) ────────────────────────────
    mean_s2   = s2.mean().sort_values(ascending=False)
    top_s2    = mean_s2.head(top_n_interactions).index.tolist()
    other_s2  = mean_s2.tail(len(mean_s2) - top_n_interactions).index.tolist()
 
    for i, col in enumerate(top_s2):
        ax2.plot(years, s2[col].values,
                 label=friendly(col),
                 color=colors10[i], lw=2)
 
    if other_s2:
        ax2.plot(years, s2[other_s2].sum(axis=1).values,
                 color="gray", lw=1.2, linestyle=":", label=f"others ({len(other_s2)})")
 
    ax2.set_ylabel("Sobol index  S_ij")
    ax2.set_title(f"2nd order — pairwise interactions  (top {top_n_interactions} shown)")
    ax2.legend(loc="upper left", ncols=2, fontsize=7)
    ax2.format(yformatter="%.3f", xminorlocator=1, ylim=(0, None))
 
    # ── Panel 3: 3rd-order (top N) ────────────────────────────────────────
    mean_s3  = s3.mean().sort_values(ascending=False)
    top_s3   = mean_s3.head(top_n_interactions).index.tolist()
    other_s3 = mean_s3.tail(len(mean_s3) - top_n_interactions).index.tolist()
 
    for i, col in enumerate(top_s3):
        ax3.plot(years, s3[col].values,
                 label=friendly(col),
                 color=colors10[i], lw=2)
 
    if other_s3:
        ax3.plot(years, s3[other_s3].sum(axis=1).values,
                 color="gray", lw=1.2, linestyle=":", label=f"others ({len(other_s3)})")
 
    ax3.set_ylabel("Sobol index  S_ijk")
    ax3.set_xlabel("Year")
    ax3.set_title(f"3rd order — triple interactions  (top {top_n_interactions} shown)")
    ax3.legend(loc="upper left", ncols=2, fontsize=7)
    ax3.format(yformatter="%.4f", xminorlocator=1, ylim=(0, None))
 
    fig.suptitle(
        f"Sobol sensitivity indices — {var_long}\n{catch_label}",
        fontsize=12, y=1.01,
    )
    fig.savefig(f"sobol_{variable}_catch{catchment}.png",
                dpi=150, bbox_inches="tight")
    return fig, axes


In [ ]:
# %% Cell S4 — Heatmap of time-mean Sobol indices (all orders at a glance)
def plot_sobol_heatmap(
    sobol_df: pd.DataFrame,
    catchment,
    variable: str = "iaf_mass",
    year = None,
):
    """
    Symmetric matrix heatmap of mean Sobol indices.
    Diagonal = 1st order.
    Off-diagonal (i,j) = 2nd order S_ij.
    A separate panel below shows the top 3rd-order triplets as a bar chart.
 
    Parameters
    ----------
    year : if given, show indices at that specific year rather than the time mean.
    """
    params   = PARAMS
    n        = len(params)
    labels   = [PARAM_LABELS.get(p, p) for p in params]
 
    s1 = sobol_df[[c for c in sobol_df if c.count("×") == 0]]
    s2 = sobol_df[[c for c in sobol_df if c.count("×") == 1]]
    s3 = sobol_df[[c for c in sobol_df if c.count("×") == 2]]
 
    if year is not None:
        t_idx  = int(np.argmin(np.abs(years - year)))
        s1_val = s1.iloc[t_idx]
        s2_val = s2.iloc[t_idx]
        s3_val = s3.iloc[t_idx]
        suffix = f" (year {int(year)})"
    else:
        s1_val = s1.mean()
        s2_val = s2.mean()
        s3_val = s3.mean()
        suffix = " (time mean)"
 
    # Build matrix
    mat = np.zeros((n, n))
    for i, p in enumerate(params):
        mat[i, i] = s1_val.get(p, 0.0)
 
    for col in s2.columns:
        p1, p2 = [c.strip() for c in col.split("×")]
        if p1 in params and p2 in params:
            i, j = params.index(p1), params.index(p2)
            mat[i, j] = mat[j, i] = s2_val.get(col, 0.0)
 
    # Figure: 2 rows — heatmap + 3rd-order bar
    fig, axes = pplt.subplots(
        nrows=2, ncols=1,
        figsize=(12, 9),
        hspace=0.5,
        height_ratios=(3, 2),
    )
    ax_heat, ax_bar = axes
 
    im = ax_heat.pcolormesh(mat, cmap="Reds", vmin=0)
    ax_heat.set_xticks(np.arange(n) + 0.5)
    ax_heat.set_yticks(np.arange(n) + 0.5)
    ax_heat.set_xticklabels(labels, rotation=30, ha="right", fontsize=8)
    ax_heat.set_yticklabels(labels, fontsize=8)
    ax_heat.set_title("1st order (diagonal) & 2nd order (off-diagonal)")
    fig.colorbar(im, ax=ax_heat, label="Sobol index")
 
    # Annotate cells
    for i in range(n):
        for j in range(n):
            ax_heat.text(j + 0.5, i + 0.5, f"{mat[i,j]:.3f}",
                         ha="center", va="center", fontsize=7,
                         color="white" if mat[i,j] > 0.5 * mat.max() else "k")
 
    # 3rd-order bar chart
    s3_sorted = s3_val.sort_values(ascending=False)
    bar_labels = [" × ".join(PARAM_LABELS.get(p.strip(), p.strip())
                              for p in col.split("×"))
                  for col in s3_sorted.index]
    colors = [c["color"] for c in itertools.islice(CYCLE, len(s3_sorted))]
    ax_bar.barh(bar_labels[::-1], s3_sorted.values[::-1], color=colors[::-1])
    ax_bar.set_xlabel("Sobol index  S_ijk")
    ax_bar.set_title("3rd order — triple interactions")
 
    var_long, _ = VAR_LABELS[variable]
    catch_label  = "all catchments" if catchment == "all" else f"catchment {catchment}"
    fig.suptitle(f"Sobol indices — {var_long}\n{catch_label}{suffix}",
                 fontsize=11, y=1.01)
    fig.savefig(f"sobol_heatmap_{variable}_catch{catchment}.png",
                dpi=150, bbox_inches="tight")
    return fig, axes


In [ ]:
# ── Example Sobol calls ─────────────────────────────────────────────────────
sobol_df = build_sobol_timeseries(catchment=11, variable="iaf_mass", max_order=3)
plot_sobol_timeseries(sobol_df, catchment=11, variable="iaf_mass")
plot_sobol_heatmap(sobol_df, catchment=11, variable="iaf_mass")
